<a href="https://colab.research.google.com/github/jasman5/UCS547-Accelerated-Data-Science/blob/main/Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

UCS547
Assignment IV
Numba Programming

In [2]:
!nvidia-smi

Thu May  7 13:18:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

1. You are given a large NumPy array of size 5000000 initialized with random values. Compute the following element-wise operation:
f(x)=x
2
+3x+5  for each element in the array and convert it into a CUDA kernel using Numba. Compare performance difference of CPU with GPU.
a. Modify the kernel to use float32 and float64

In [3]:
import numpy as np
import time
from numba import cuda

N = 5000000
# Random array
arr32 = np.random.rand(N).astype(np.float32)
arr64 = np.random.rand(N).astype(np.float64)

#cpu
def cpu_compute(arr):
    return arr * arr + 3 * arr + 5

#gpu kernel float32
@cuda.jit
def gpu_compute_float32(inp, out):
    i = cuda.grid(1)

    if i < inp.size:
        out[i] = inp[i] * inp[i] + 3 * inp[i] + 5

#gpu kernel float64
@cuda.jit
def gpu_compute_float64(inp, out):
    i = cuda.grid(1)

    if i < inp.size:
        out[i] = inp[i] * inp[i] + 3 * inp[i] + 5

#cpu time
start = time.time()
cpu_result = cpu_compute(arr32)
end = time.time()
print("CPU Time:", end - start)

#gpu float32
d_in32 = cuda.to_device(arr32)
d_out32 = cuda.device_array_like(arr32)
threads = 256
blocks = (N + threads - 1) // threads
start = time.time()
gpu_compute_float32[blocks, threads](d_in32, d_out32)
cuda.synchronize()
end = time.time()
print("GPU float32 Time:", end - start)

#gpu float64
d_in64 = cuda.to_device(arr64)
d_out64 = cuda.device_array_like(arr64)
start = time.time()
gpu_compute_float64[blocks, threads](d_in64, d_out64)
cuda.synchronize()
end = time.time()
print("GPU float64 Time:", end - start)

CPU Time: 0.022166967391967773
GPU float32 Time: 2.10280442237854
GPU float64 Time: 0.07676553726196289


2. Implement and benchmark a 1-D histogram computation for 1 million random values in Python using Numba. Compare different approaches (pure Python, NumPy, and Numba-accelerated) and analyze performance and correctness.

In [4]:
import numpy as np
import time
from numba import njit

N = 1000000
data = np.random.randint(0, 100, N)

# python
def python_histogram(arr, bins):
    hist = [0] * bins
    for value in arr:
        hist[value] += 1
    return hist

# numpy
def numpy_histogram(arr):
    return np.histogram(arr, bins=100, range=(0,100))[0]

# numba
@njit
def numba_histogram(arr, bins):
    hist = np.zeros(bins, dtype=np.int32)
    for i in range(arr.size):
        hist[arr[i]] += 1
    return hist

#PYTHON TIMING
start = time.time()
h1 = python_histogram(data, 100)
end = time.time()
print("Pure Python Time:", end - start)

#NUMPY TIMING
start = time.time()
h2 = numpy_histogram(data)
end = time.time()
print("NumPy Time:", end - start)

#NUMBA TIMING
start = time.time()
h3 = numba_histogram(data, 100)
end = time.time()
print("Numba Time:", end - start)

#CORRECTNESS
print("Python vs NumPy:", np.all(np.array(h1) == h2))
print("NumPy vs Numba:", np.all(h2 == h3))

Pure Python Time: 0.11166977882385254
NumPy Time: 0.018611907958984375
Numba Time: 1.0122530460357666
Python vs NumPy: True
NumPy vs Numba: True


3. Write a function monte_carlo_pi(nsamples) that estimates the value of π by generating random x, y coordinates between 0 and 1 and checking if they fall inside a unit circle.
a. Implement the function in pure Python first and later create a Numba version.

In [5]:
import random
import time
from numba import njit

#PURE PYTHON
def monte_carlo_pi(nsamples):

    inside = 0

    for i in range(nsamples):
        x = random.random()
        y = random.random()

        if x*x + y*y < 1:
            inside += 1

    return 4 * inside / nsamples

#NUMBA VERSION
@njit
def monte_carlo_pi_numba(nsamples):

    inside = 0

    for i in range(nsamples):
        x = np.random.random()
        y = np.random.random()

        if x*x + y*y < 1:
            inside += 1

    return 4 * inside / nsamples

b. Program a script to compare the execution time for 5 million samples. Report the Speedup Factor (Python Time / Numba Time).

In [6]:

samples = 5000000

# PYTHON TIMING
start = time.time()

pi1 = monte_carlo_pi(samples)

end = time.time()

python_time = end - start

print("Python PI:", pi1)
print("Python Time:", python_time)

# NUMBA TIMING
start = time.time()

pi2 = monte_carlo_pi_numba(samples)

end = time.time()

numba_time = end - start

print("Numba PI:", pi2)
print("Numba Time:", numba_time)

#SPEEDUP
print("Speedup Factor:",
      python_time / numba_time)

Python PI: 3.1405744
Python Time: 0.8787157535552979
Numba PI: 3.1420936
Numba Time: 0.1759171485900879
Speedup Factor: 4.995054550382869


c. Why does the very first execution of the Numba function take slightly longer than the second execution?
Answer

The first execution includes:

Compilation of Python code into optimized machine code.
Type inference and optimization.

After compilation, the compiled version is reused, making later executions faster.

4. You have a 1D NumPy array representing pixel intensities (values 0–255). You need to increase the brightness of every pixel by 20%, but ensure no value exceeds 255.

a. Write a function adjust_brightness(pixel_value) using the @vectorize decorator.

b. Apply this function to an array of 10 million random integers.

c. Change the decorator to @vectorize(['int64(int64)'], target='parallel'). Measure the time difference when the work is automatically split across your CPU cores.

In [7]:
import numpy as np
import time
from numba import vectorize

N = 10000000
pixels = np.random.randint(0, 256, N)

# NORMAL VECTORIZE
@vectorize(['int64(int64)'])
def adjust_brightness(x):

    value = int(x * 1.2)
    if value > 255:
        return 255
    return value

start = time.time()
result1 = adjust_brightness(pixels)
end = time.time()
print("Normal Vectorize Time:",
      end - start)

# PARALLEL VERSION
@vectorize(['int64(int64)'], target='parallel')
def adjust_brightness_parallel(x):

    value = int(x * 1.2)
    if value > 255:
        return 255
    return value

start = time.time()
result2 = adjust_brightness_parallel(pixels)
end = time.time()
print("Parallel Vectorize Time:",
      end - start)

Normal Vectorize Time: 0.024593591690063477
Parallel Vectorize Time: 0.021511554718017578


d. What happens if you try to pass a list instead of a NumPy array to this function?If a Python list is passed:

Numba automatically converts it internally to a NumPy array.
Execution may become slower because conversion overhead is added.

5. Write Python code to generate synthetic training data of 100,000 samples, 10 features and binary labels {-1, +1}. Implement binary logistic regression using the mathematical formula for gradient descent:

a. Using standard NumPy (without Numba)
b. Using Numba JIT acceleration
c. Compare correctness and performance.

In [8]:
import numpy as np
import time
from numba import njit

#DATA
samples = 100000
features = 10
X = np.random.randn(samples, features)
y = np.random.choice([-1, 1], samples)
learning_rate = 0.01
epochs = 100

#NUMPY
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def logistic_numpy(X, y):
    weights = np.zeros(X.shape[1])

    for epoch in range(epochs):
        z = np.dot(X, weights)
        predictions = sigmoid(z)
        gradient = np.dot(X.T,
                          (predictions - y)) / len(y)
        weights -= learning_rate * gradient
    return weights

#NUMBA
@njit
def logistic_numba(X, y):
    weights = np.zeros(X.shape[1])

    for epoch in range(epochs):
        z = np.dot(X, weights)
        predictions = 1 / (1 + np.exp(-z))
        gradient = np.dot(X.T,
                         (predictions - y)) / len(y)
        weights -= learning_rate * gradient
    return weights

#NUMPY TIMING
start = time.time()
w1 = logistic_numpy(X, y)
end = time.time()
numpy_time = end - start
print("NumPy Time:", numpy_time)

#NUMBA TIMING
start = time.time()
w2 = logistic_numba(X, y)
end = time.time()
numba_time = end - start
print("Numba Time:", numba_time)

#CORRECTNESS
print("Weights Similar:",
      np.allclose(w1, w2))

print("Speedup:",
      numpy_time / numba_time)

NumPy Time: 0.4542856216430664
Numba Time: 1.6308636665344238
Weights Similar: True
Speedup: 0.27855524098370577


6. Write a CUDA kernel to add two large matrices (A + B = C) of size 1024 × 1024.

In [9]:
from numba import cuda
import numpy as np

N = 1024

# CUDA KERNEL
@cuda.jit
def matrix_add(A, B, C):
    row, col = cuda.grid(2)

    if row < A.shape[0] and col < A.shape[1]:
        C[row][col] = A[row][col] + B[row][col]

# MATRICES
A = np.random.randint(0, 100, (N, N))
B = np.random.randint(0, 100, (N, N))
C = np.zeros((N, N))

# DEVICE MEMORY
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((N, N))

threads = (16, 16)
blocks_x = (N + threads[0] - 1) // threads[0]
blocks_y = (N + threads[1] - 1) // threads[1]
blocks = (blocks_x, blocks_y)

# KERNEL LAUNCH
matrix_add[blocks, threads](d_A, d_B, d_C)
C = d_C.copy_to_host()

print("Matrix Addition Completed")

Matrix Addition Completed
